In [ ]:
# ============================================================
# STEP 1 : Import Libraries
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)

In [ ]:
from pathlib import Path
import os

def _find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "README.md").exists() and (candidate / "notebooks").is_dir():
            return candidate
    return current

PROJECT_ROOT = _find_project_root()
DATA_ROOT = Path(os.environ.get("GANGA_DATA_ROOT", PROJECT_ROOT / "data")).expanduser()
OUTPUT_ROOT = Path(os.environ.get("GANGA_OUTPUT_ROOT", PROJECT_ROOT / "outputs")).expanduser()

PROJECT_DIR = PROJECT_ROOT
OUTPUT_DIR = OUTPUT_ROOT / "cwc"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# ============================================================
# STEP 3 : Load Datasets
# ============================================================

bio = pd.read_csv(bio_file)

chem = pd.read_csv(chem_file)

phys = pd.read_csv(phys_file)

print("Biological :", bio.shape)
print("Chemical   :", chem.shape)
print("Physical   :", phys.shape)

In [ ]:
# ============================================================
# STEP 4 : Standardize Date Column
# ============================================================

for df in [bio, chem, phys]:

    df["Data Acquisition Time"] = pd.to_datetime(
        df["Data Acquisition Time"]
    )

In [ ]:
# ============================================================
# STEP 5 : Date Validation
# ============================================================

print(bio["Data Acquisition Time"].dtype)
print(chem["Data Acquisition Time"].dtype)
print(phys["Data Acquisition Time"].dtype)

In [ ]:
# ============================================================
# STEP 6 : Merge Keys
# ============================================================

KEYS = [
    "Station_UID",
    "Data Acquisition Time"
]

In [ ]:
# ============================================================
# STEP 7 : Duplicate Key Check
# ============================================================

print("Bio Duplicate Keys  :",
      bio.duplicated(KEYS).sum())

print("Chem Duplicate Keys :",
      chem.duplicated(KEYS).sum())

print("Phys Duplicate Keys :",
      phys.duplicated(KEYS).sum())

In [ ]:
# ============================================================
# STEP 8 : Inspect Duplicate Biological Keys
# ============================================================

bio_dup = (
    bio[
        bio.duplicated(KEYS, keep=False)
    ]
    .sort_values(KEYS)
)

print("Duplicate Rows :", len(bio_dup))

display(bio_dup.head(20))

In [ ]:
# ============================================================
# STEP 9 : Duplicate Groups
# ============================================================

dup_groups = (
    bio_dup
    .groupby(KEYS)
    .size()
    .reset_index(name="Count")
    .sort_values("Count", ascending=False)
)

display(dup_groups)

print("Duplicate Groups :", len(dup_groups))

In [ ]:
# ============================================================
# STEP 10 : Compare Duplicate Rows
# ============================================================

for _, grp in bio_dup.groupby(KEYS):

    print("=" * 100)
    print(grp[KEYS].iloc[0].to_dict())

    diff = {}

    for col in grp.columns:

        if col in KEYS:
            continue

        vals = grp[col].astype(str).unique()

        if len(vals) > 1:
            diff[col] = vals.tolist()

    print(diff)

In [ ]:
# ============================================================
# STEP 11 : Remove Source_File Duplicates
# ============================================================

bio = (
    bio
    .drop_duplicates(
        subset=[
            c for c in bio.columns
            if c != "Source_File"
        ]
    )
    .reset_index(drop=True)
)

print("Biological Shape :", bio.shape)

print(
    "Duplicate Keys :",
    bio.duplicated(KEYS).sum()
)

In [ ]:
# ============================================================
# STEP 12 : Common Metadata Columns
# ============================================================

metadata_cols = [
    "Station_UID",
    "Station",
    "Latitude",
    "Longitude",
    "State",
    "District",
    "Tehsil",
    "Basin",
    "Data Acquisition Time",
    "Source_File"
]

In [ ]:
# ============================================================
# STEP 13 : Keep Only Chemical Parameters
# ============================================================

chem_params = chem[
    KEYS +
    [
        c for c in chem.columns
        if c not in metadata_cols
        and c not in KEYS
    ]
]

In [ ]:
phys_params = phys[
    KEYS +
    [
        c for c in phys.columns
        if c not in metadata_cols
        and c not in KEYS
    ]
]

In [ ]:
master = bio.merge(
    chem_params,
    on=KEYS,
    how="left"
)

master = master.merge(
    phys_params,
    on=KEYS,
    how="left"
)

In [ ]:
print("Biological rows :", len(bio))
print("Master rows     :", len(master))
print("Duplicate Keys  :", master.duplicated(KEYS).sum())
print("Missing UID     :", master["Station_UID"].isna().sum())
print("Total Columns   :", len(master.columns))

In [ ]:
# ============================================================
# STEP 18 : Final Validation Report
# ============================================================

print("=" * 60)
print("FINAL CWC MASTER DATASET")
print("=" * 60)

print(f"Rows                : {len(master):,}")
print(f"Columns             : {len(master.columns)}")

print(f"\nStations            : {master['Station_UID'].nunique()}")

print(
    f"Date Range          : "
    f"{master['Data Acquisition Time'].min()}  -->  "
    f"{master['Data Acquisition Time'].max()}"
)

print(f"\nDuplicate Keys      : {master.duplicated(KEYS).sum()}")

print(f"Missing Station_UID : {master['Station_UID'].isna().sum()}")

print("=" * 60)

In [ ]:
# ============================================================
# STEP 19 : Save Final Master Dataset
# ============================================================

final_file = OUTPUT_DIR / "Final_Ganga_CWC_Master.csv"

master.to_csv(
    final_file,
    index=False
)

print("Saved Successfully")
print(final_file)

In [ ]:
# ============================================================
# STEP 20 : Missing Value Summary
# ============================================================

missing = (
    master.isna()
          .mean()
          .mul(100)
          .round(2)
          .sort_values(ascending=False)
)

missing = missing[missing > 0]

display(missing)